In [1]:
%load_ext autoreload
%autoreload 2
import sys, time
sys.path.append(".")
print(" Ready")

 Ready


In [2]:
from graph.build_graph import build_graph

app = build_graph()  # no checkpointer/interrupt for this run — full autonomous end-to-end

test_queries = [
    "Compare the battery recycling strategies of Tesla, Redwood Materials, and CATL",
    "What causes lithium plating in fast-charging EV batteries?",
    "Is EV battery technology actually improving fast enough to matter?",  # deliberately vague/opinion-adjacent
]

run_results = []
for q in test_queries:
    print(f"\n{'='*70}\nQUERY: {q}\n{'='*70}")
    initial_state = {
        "query": q, "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    start = time.time()
    result = app.invoke(initial_state)
    elapsed = time.time() - start
    run_results.append({"query": q, "result": result, "elapsed": elapsed})
    print(f"\n{elapsed:.1f}s | iterations: {result['iteration_count']} | errors: {len(result['errors'])}")
    print(f"Report length: {len(result['final_report']) if result['final_report'] else 0} chars")


QUERY: Compare the battery recycling strategies of Tesla, Redwood Materials, and CATL


2026-10-30 17:27:28,449 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


⚠️ Gemini unavailable (ClientError), falling back to Tavily...⚠️ Gemini unavailable (ClientError), falling back to Tavily...



2026-10-30 17:27:32,004 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-30 17:27:32,802 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


... (13 outputs left out)

2026-10-30 17:32:08,930 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-30 17:32:09,142 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-10-30 17:32:11,391 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3 retries=0
2026-10-30 17:32:11,783 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=2 retries=0
2026-10-30 17:32:11,789 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-10-30 17:32:43,390 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What is the current annual rate of improvement in EV battery energy density and cost per kilowatt-hour? facts_found=9
2026-10-30 17:32:55,163 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do these technological improvements translate into measurable changes in vehicle range and charging speed for consumers? facts_found=4
2026-10-30 17:32:55,494 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the projected timelines for next-generation battery


175.1s | iterations: 6 | errors: 0
Report length: 6609 chars


In [3]:
vague_result = run_results[2]["result"]
print("Plan generated for the vague query:")
for q in vague_result["plan"]:
    print(" -", q)
print(f"\nErrors: {vague_result['errors']}")
print(f"\nFirst 500 chars of report:\n{vague_result['final_report'][:500]}")

Plan generated for the vague query:
 - What is the current annual rate of improvement in EV battery energy density and cost per kilowatt-hour?
 - How do these technological improvements translate into measurable changes in vehicle range and charging speed for consumers?
 - What are the projected timelines for next-generation battery chemistries to reach commercial viability and market scale?
 - How do current battery degradation rates and second-life applications impact the long-term economic and environmental value of EVs?

Errors: []

First 500 chars of report:
# Is EV Battery Technology Actually Improving Fast Enough to Matter?

The rapid evolution of electric vehicle (EV) battery technology is a central factor in the adoption of sustainable transportation. To determine whether these advancements are significant enough to impact consumer behavior and market dynamics, it is necessary to examine current rates of improvement, their translation into vehicle performance, future chemical 

In [4]:
print(f"{'Query':<55} {'Time':<8} {'Iters':<6} {'Errors':<7} {'Report'}")
for r in run_results:
    print(f"{r['query'][:52]:<55} {r['elapsed']:.1f}s{'':<4} {r['result']['iteration_count']:<6} "
          f"{len(r['result']['errors']):<7} {len(r['result']['final_report']) if r['result']['final_report'] else 0} chars")

Query                                                   Time     Iters  Errors  Report
Compare the battery recycling strategies of Tesla, R    92.4s     6      0       5003 chars
What causes lithium plating in fast-charging EV batt    162.1s     6      0       7110 chars
Is EV battery technology actually improving fast eno    175.1s     6      0       6609 chars


In [ ]:
review = """# Week 3 Checkpoint (Day 21)

## Tested
3 end-to-end queries through the full autonomous graph: a 3-way comparison
(battery recycling), a narrow technical question (lithium plating causes),
and a deliberately vague/opinion-adjacent query (is battery tech improving
fast enough).

## Results
| Query | Time | Iterations | Errors | Report |
|---|---|---|---|---|
| Recycling comparison | 92.4s | 6 | 0 | 5003 chars (3 summaries, not 4 — see issue below) |
| Lithium plating | 162.1s | 6 | 0 | 7110 chars (4 summaries, mostly correctly hedged UNVERIFIED) |
| Vague/opinion query | 175.1s | 6 | 0 | 6609 chars (4 summaries, correctly mostly UNVERIFIED) |

Timing climbed steadily across the session (92s -> 162s -> 175s), consistent
with accumulating pressure against Groq's rolling 60s output-token budget
across consecutive runs in one sitting.

## Issue found: sub-topics silently vanish when extraction finds zero facts
Query 1's plan had 4 sub-questions. One ("environmental impacts... carbon
footprint") returned facts_found=0 at extraction. Because summarization_agent
groups by sub_question keys PRESENT in fact_check_results, and that
sub-question never produced any fact_check_results entries, it never
appeared in summarization at all -- not even as a "no information found"
summary. The Day 11 empty-facts fallback only protects against an empty
LIST being passed in, not a sub-topic being entirely ABSENT. Net effect:
the final report silently covers 3 of 4 planned angles with no indication
a 4th was ever attempted.

This is a real gap, not a crash -- nothing errored, which is exactly why
it went unnoticed until this checkpoint. Candidate fix (not done today):
summarization_node should iterate over the ORIGINAL plan, not just the
keys present in fact_check_results, and explicitly emit a "no information
found" summary for any planned sub-question with zero results.

## Good signals
- Query 3 (vague/opinion query): Supervisor correctly decomposed a subjective
  question into concrete, researchable sub-questions rather than inviting
  ungrounded opinion. Resulting confidence breakdown was mostly UNVERIFIED --
  an honest reflection of the question's inherent uncertainty, not false
  confidence.
- Query 3 also hit a real mid-run RateLimitError that was fully absorbed by
  the fail-safe -- errors: [] in the final state despite the failure,
  confirming Day 18's fix holds under genuinely new, unplanned load.

## Week 3 recap
- Day 15: benchmarked parallel vs sequential search (2-2.8x speedup)
- Day 16: verified LangGraph reducer/Send mechanics against real API calls
- Day 17: shipped Send-based parallel web search + rate-limit retry
- Day 18: centralized retry logic in call_llm, fixed downstream rate-limit gap
- Day 19: DuckDuckGo fallback tier + concurrency semaphore + DEMO_MODE
  (per mentor feedback), human-in-the-loop interrupt_before
- Day 20: SqliteSaver persistent checkpointing, verified across simulated
  process restarts and independent threads
- Day 21: 3 real end-to-end stress-test queries; found and documented the
  silent-sub-topic-drop gap above

## Carried into Week 4
- The silent-drop gap above is a good candidate for Day 22's evaluation
  harness: "does the report account for every planned sub-question" is a
  concrete, testable quality criterion, not just a one-off bug to patch.
- Groq's 1000 output-token/min limit remains the binding constraint under
  real multi-query sessions; fail-safes handle it correctly but it's worth
  keeping in mind for demo pacing (per mentor's earlier feedback).
"""
with open("docs/week3_review.md", "w", encoding="utf-8") as f:
    f.write(review)
print(" docs/week3_review.md written")